In [ ]:
import argparse
import os
import sys
import math
from itertools import cycle
from typing import Union

sys.path.append(os.getcwd())


import h5py
import cv2
import torch
import numpy as np
import openslide

from preprocess.create_coords import knn, fps


data_root = '/data/smb/syh/WSI_cls/'

def patch_coords(h5_file):
    with h5py.File(h5_file, 'r') as f:
            return torch.tensor(f['coords'])

def hex_to_bgr(hex_color):
    # 去掉十六进制颜色前缀‘#’
    hex_color = hex_color.lstrip('#')
    # 解析十六进制颜色值，并转换为整数
    rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    # 调整颜色顺序从RGB到BGR
    bgr = (rgb[2], rgb[1], rgb[0])
    return bgr

def read_tif(img_path):
    wsi = openslide.OpenSlide(img_path)

    level = 0
    for lvl, dims in enumerate(wsi.level_dimensions):
        level = lvl
        if max(dims) < 8000:
            break

    # print(len(wsi.level_dimensions))
    dims = wsi.level_dimensions[level]
    downsample_factor = wsi.level_downsamples[level]

    img_org = wsi.read_region((0, 0), level, dims).convert('RGB')
    return np.array(img_org, dtype=np.uint8), downsample_factor

def draw_points(img_path, coords, k):
    colors = [
        '#FFFF08',
        '#00FF00',
        '#08FFFF',
        '#FF00FF',
        '#FF0000',
        '#0808FF',
    ]

    img_org, downsample_factor = read_tif(img_path) #RGB
    img_org = img_org[:, :, ::-1] #BGR

    img = img_org

    overlay = np.zeros(img.shape, dtype=np.uint8)
    num_instances = coords.shape[0]
    for idx, color in zip(range(0, num_instances, k), cycle(colors)):
        sub_coord = coords[idx : (idx + k)]

        for coord in sub_coord:
            coord = coord / downsample_factor
            coord = coord.tolist()
            coord = [int(c) for c in coord]
            coord2 = [int(c + 512 / downsample_factor) for c in coord]
            # print(coord, hex_to_bgr(color))
            # cv2.circle(img, coord, 3, hex_to_bgr(color), 3)
            cv2.rectangle(overlay, coord, coord2, hex_to_bgr(color), -1)

    # 将overlay层与原图合并
    alpha = 0.8
    img = cv2.addWeighted(img, alpha, overlay, 1 - alpha, 0)
    return img, img_org

def showArrayOrTensor(t:Union[np.ndarray|torch.Tensor],tag=""):
    if isinstance(t,torch.Tensor):
        print(f"tag:{tag}, size:{t.shape}, type:{t.dtype}, device:{t.device}")
    elif isinstance(t,np.ndarray):
        print(f"tag:{tag}, size:{t.shape}, type:{t.dtype}")
    else:
        print(f"Tag({tag}) is {type(t)}, it's not a array or tensor!")
    
def main():

    parser = argparse.ArgumentParser()
    # Required parameters
    parser.add_argument("--data_root", default='/data/smb/syh/WSI_cls/')
    parser.add_argument('--dataset', default='cam16', type=str)
    parser.add_argument('--k', default=64, type=int)
    args = parser.parse_args()

    assert args.dataset in ['cam16', 'brac', 'lung']

    h5_dir = os.path.join(data_root, args.dataset, 'feat_clam_resnet50', 'patch_size_256_at_mag_20', 'h5_files')
    img_dir = os.path.join(data_root, args.dataset, 'img')

    os.makedirs('tmp', exist_ok=True)

    for slide in os.listdir(img_dir):

        slide_id, _ = os.path.splitext(slide)

        h5_path = os.path.join(h5_dir, slide_id + '.h5')
        coords = patch_coords(h5_path)

        idx = fps(coords, math.ceil(len(coords) / args.k))
        idx = knn(coords, idx, k=args.k)

        coords = coords[idx]

        img_path = os.path.join(img_dir, slide)
        img, img_org = draw_points(img_path, coords, args.k)

        showArrayOrTensor(img,'img') 
        showArrayOrTensor(idx,'idx')
        showArrayOrTensor(coords,'coords')
        showArrayOrTensor(slide,'slide')
        print( h5_path)
        
        input("pause")


        cv2.imwrite('tmp/{}.jpg'.format(slide_id), img)
        cv2.imwrite('tmp/{}_org.jpg'.format(slide_id), img_org)

#main()

In [6]:
import os
import random
import torch
import pandas as pd

import torch.utils.data as data

class WSIDataset(data.Dataset):
    def __init__(self, data_root, state, fold, dataset):
        super().__init__()
        # Set all input args as attributes
        self.data_root = data_root

        data_set = state

        self.feat_dir = os.path.join(
            self.data_root,
                # 'feat_clam_resnet50',
                'patch_size_256_at_mag_20',
                'pt_files'
            )

        self.idx = os.path.join(
            self.data_root,
            "idx_64"
        )

        self.pos = os.path.join(
            self.data_root,
            'pos'
        )

        # dataset_name = os.path.basename(self.data_root)
        dataset_name = dataset
        self.file_list_csv = os.path.join('dataset_csv', 'dataset_list', dataset_name, '{}.csv'.format(dataset_name))
        self.split_dir = os.path.join('dataset_csv', 'splits',  dataset_name)
        wsis = self.get_wsis(data_set, fold)
        self.wsis = wsis
        self.slide_names = list(self.wsis.keys())

        self.data_set = data_set

        self.tmp_data = {}

        # for idx in range(len(self)):
        #     slide_id = self.slide_names[idx]
        #     # label = self.wsis[slide_id]
        #     full_path = os.path.join(self.feat_dir, '{}.pt'.format(slide_id))
        #     self.get_data(full_path)



    def get_wsis(self, data_set, fold):
        file_list = pd.read_csv(self.file_list_csv)
        file_list['slide_id'] = file_list['slide_id'].apply(
            lambda x: os.path.splitext(x)[0]
        )

        split_file = os.path.join(self.split_dir, 'splits_{}.csv'.format(fold))
        splits = pd.read_csv(split_file)
        train_split = splits['train'].dropna()
        val_split = splits['val'].dropna()
        test_split = splits['test'].dropna()

        if data_set == 'train':
            mask1 = file_list['slide_id'].isin(train_split)
            mask2 = file_list['slide_id'].isin(val_split)
            slide_ids = file_list['slide_id'][mask1 | mask2]
        else:
            mask1 = file_list['slide_id'].isin(test_split)
            slide_ids = file_list['slide_id'][mask1]


        # slide_ids = slide_ids[:3]
        wsis = {}
        for slide_id in slide_ids:
            mask = file_list['slide_id'] == slide_id
            label = int(file_list[mask].iloc[0]['label'])
            wsis[slide_id] = label

        return wsis

    def dict2tensor(self, features):
        assert isinstance(features, dict)
        return torch.tensor(list(features.values()))

    def get_data(self, full_path):
        features = self.tmp_data.get(full_path, None)
        # print('start loading...')

        if features is None:
            features = torch.load(full_path)
            if isinstance(features, dict):
                features = self.dict2tensor(features)
            self.tmp_data[full_path] = features

        # print('done loading.......')
        return features

    def __len__(self):
        return len(self.wsis.keys())

    def __getitem__(self, idx):
        slide_id = self.slide_names[idx]
        label = self.wsis[slide_id]
        full_path = os.path.join(self.feat_dir, '{}.pt'.format(slide_id))
        # print(full_path)

        # features = torch.load(full_path)
        features = self.get_data(full_path)


        # idx = torch.load(os.path.join(self.idx, slide_id + '.pt')).long()
        pos = torch.load(os.path.join(self.pos, slide_id + '.pt'))


        pos_len = pos.shape[0]
        feat_len = features.shape[0]
        if pos_len > feat_len:
            # print('>>')
            pos = pos[:feat_len]
        else:
            # print('<<')
            # print('before')
            # print(pos.shape, pos[:feat_len - pos_len].shape, pos_len, feat_len)
            # pos = torch.cat([pos, pos[:feat_len - pos_len]], dim=0)

            repeat_times = feat_len // pos_len
            remainder = feat_len % pos_len

            # 重复 pos 并拼接
            pos = torch.cat([pos] * repeat_times + [pos[:remainder]], dim=0)
            # print(pos.shape)

        features = features.clone()
        # print(os.path.join(self.idx, slide_id + '.pt'))
        # print(full_path)
        # print(pos_len, feat_len)


        # features = features[idx]
        # pos = pos[idx]

        # print(full_path)
        # print(os.path.join(self.pos, slide_id + '.pt'))
        # print(pos.shape, features.shape)

        # print(pos.shape, features.shape)
        assert pos.shape[0] == features.shape[0]
        if self.data_set == 'train':
            index = [x for x in range(features.shape[0])]
            random.shuffle(index)
            features = features[index]
            pos = pos[index]

        assert pos.shape[0] == features.shape[0]
        return features, pos, label,slide_id


In [31]:
from models.model_interface import ModelInterface
import os
import os
import random
import torch
import pandas as pd

import torch.utils.data as data

import argparse
import os
import sys
import math
from itertools import cycle
from typing import Union
from pytorch_grad_cam import GradCAM
sys.path.append(os.getcwd())


import h5py
import cv2
import torch
import numpy as np
import openslide

from preprocess.create_coords import knn, fps


data_root = '/data/smb/syh/WSI_cls/'

def patch_coords(h5_file):
    with h5py.File(h5_file, 'r') as f:
            return torch.tensor(f['coords'])

def hex_to_bgr(hex_color):
    # 去掉十六进制颜色前缀‘#’
    hex_color = hex_color.lstrip('#')
    # 解析十六进制颜色值，并转换为整数
    rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    # 调整颜色顺序从RGB到BGR
    bgr = (rgb[2], rgb[1], rgb[0])
    return bgr

def read_tif(img_path):
    wsi = openslide.OpenSlide(img_path)

    level = 0
    for lvl, dims in enumerate(wsi.level_dimensions):
        level = lvl
        if max(dims) < 8000:
            break

    # print(len(wsi.level_dimensions))
    dims = wsi.level_dimensions[level]
    downsample_factor = wsi.level_downsamples[level]

    img_org = wsi.read_region((0, 0), level, dims).convert('RGB')
    return np.array(img_org, dtype=np.uint8), downsample_factor

def draw_points(img_path, coords, k):
    colors = [
        '#FFFF08',
        '#00FF00',
        '#08FFFF',
        '#FF00FF',
        '#FF0000',
        '#0808FF',
    ]

    img_org, downsample_factor = read_tif(img_path) #RGB
    img_org = img_org[:, :, ::-1] #BGR

    img = img_org

    overlay = np.zeros(img.shape, dtype=np.uint8)
    num_instances = coords.shape[0]
    for idx, color in zip(range(0, num_instances, k), cycle(colors)):
        sub_coord = coords[idx : (idx + k)]

        for coord in sub_coord:
            coord = coord / downsample_factor
            coord = coord.tolist()
            coord = [int(c) for c in coord]
            coord2 = [int(c + 512 / downsample_factor) for c in coord]
            # print(coord, hex_to_bgr(color))
            # cv2.circle(img, coord, 3, hex_to_bgr(color), 3)
            cv2.rectangle(overlay, coord, coord2, hex_to_bgr(color), -1)

    # 将overlay层与原图合并
    alpha = 0.8
    img = cv2.addWeighted(img, alpha, overlay, 1 - alpha, 0)
    return img, img_org

def showArrayOrTensor(t:Union[np.ndarray|torch.Tensor],tag=""):
    if isinstance(t,torch.Tensor):
        print(f"tag:{tag}, size:{t.shape}, type:{t.dtype}, device:{t.device}")
    elif isinstance(t,np.ndarray):
        print(f"tag:{tag}, size:{t.shape}, type:{t.dtype}")
    else:
        print(f"Tag({tag}) is {type(t)}, it's not a array or tensor!")
    
def main():

    parser = argparse.ArgumentParser()
    # Required parameters
    parser.add_argument("--data_root", default='/data/smb/syh/WSI_cls/')
    parser.add_argument('--dataset', default='cam16', type=str)
    parser.add_argument('--k', default=64, type=int)
    args = parser.parse_args()

    assert args.dataset in ['cam16', 'brac', 'lung']

    h5_dir = os.path.join(data_root, args.dataset, 'feat_clam_resnet50', 'patch_size_256_at_mag_20', 'h5_files')
    img_dir = os.path.join(data_root, args.dataset, 'img')

    os.makedirs('tmp', exist_ok=True)

    for slide in os.listdir(img_dir):

        slide_id, _ = os.path.splitext(slide)

        h5_path = os.path.join(h5_dir, slide_id + '.h5')
        coords = patch_coords(h5_path)

        idx = fps(coords, math.ceil(len(coords) / args.k))
        idx = knn(coords, idx, k=args.k)

        coords = coords[idx]

        img_path = os.path.join(img_dir, slide)
        img, img_org = draw_points(img_path, coords, args.k)

        showArrayOrTensor(img,'img') 
        showArrayOrTensor(idx,'idx')
        showArrayOrTensor(coords,'coords')
        showArrayOrTensor(slide,'slide')
        print( h5_path)
        
        input("pause")


        cv2.imwrite('tmp/{}.jpg'.format(slide_id), img)
        cv2.imwrite('tmp/{}_org.jpg'.format(slide_id), img_org)

#main()
class WSIDataset(data.Dataset):
    def __init__(self, data_root, state, fold, dataset):
        super().__init__()
        # Set all input args as attributes
        self.data_root = data_root

        data_set = state

        self.feat_dir = os.path.join(
            self.data_root,
                # 'feat_clam_resnet50',
                'patch_size_256_at_mag_20',
                'pt_files'
            )

        self.idx = os.path.join(
            self.data_root,
            "idx_64"
        )

        self.pos = os.path.join(
            self.data_root,
            'pos'
        )

        # dataset_name = os.path.basename(self.data_root)
        dataset_name = dataset
        self.file_list_csv = os.path.join('dataset_csv', 'dataset_list', dataset_name, '{}.csv'.format(dataset_name))
        self.split_dir = os.path.join('dataset_csv', 'splits',  dataset_name)
        wsis = self.get_wsis(data_set, fold)
        self.wsis = wsis
        self.slide_names = list(self.wsis.keys())

        self.data_set = data_set

        self.tmp_data = {}

        # for idx in range(len(self)):
        #     slide_id = self.slide_names[idx]
        #     # label = self.wsis[slide_id]
        #     full_path = os.path.join(self.feat_dir, '{}.pt'.format(slide_id))
        #     self.get_data(full_path)



    def get_wsis(self, data_set, fold):
        file_list = pd.read_csv(self.file_list_csv)
        file_list['slide_id'] = file_list['slide_id'].apply(
            lambda x: os.path.splitext(x)[0]
        )

        split_file = os.path.join(self.split_dir, 'splits_{}.csv'.format(fold))
        splits = pd.read_csv(split_file)
        train_split = splits['train'].dropna()
        val_split = splits['val'].dropna()
        test_split = splits['test'].dropna()

        if data_set == 'train':
            mask1 = file_list['slide_id'].isin(train_split)
            mask2 = file_list['slide_id'].isin(val_split)
            slide_ids = file_list['slide_id'][mask1 | mask2]
        else:
            mask1 = file_list['slide_id'].isin(test_split)
            slide_ids = file_list['slide_id'][mask1]


        # slide_ids = slide_ids[:3]
        wsis = {}
        for slide_id in slide_ids:
            mask = file_list['slide_id'] == slide_id
            label = int(file_list[mask].iloc[0]['label'])
            wsis[slide_id] = label

        return wsis

    def dict2tensor(self, features):
        assert isinstance(features, dict)
        return torch.tensor(list(features.values()))

    def get_data(self, full_path):
        features = self.tmp_data.get(full_path, None)
        # print('start loading...')

        if features is None:
            features = torch.load(full_path)
            if isinstance(features, dict):
                features = self.dict2tensor(features)
            self.tmp_data[full_path] = features

        # print('done loading.......')
        return features

    def __len__(self):
        return len(self.wsis.keys())

    def __getitem__(self, idx):
        slide_id = self.slide_names[idx]
        label = self.wsis[slide_id]
        full_path = os.path.join(self.feat_dir, '{}.pt'.format(slide_id))
        # print(full_path)

        # features = torch.load(full_path)
        features = self.get_data(full_path)


        # idx = torch.load(os.path.join(self.idx, slide_id + '.pt')).long()
        pos = torch.load(os.path.join(self.pos, slide_id + '.pt'))


        pos_len = pos.shape[0]
        feat_len = features.shape[0]
        if pos_len > feat_len:
            # print('>>')
            pos = pos[:feat_len]
        else:
            # print('<<')
            # print('before')
            # print(pos.shape, pos[:feat_len - pos_len].shape, pos_len, feat_len)
            # pos = torch.cat([pos, pos[:feat_len - pos_len]], dim=0)

            repeat_times = feat_len // pos_len
            remainder = feat_len % pos_len

            # 重复 pos 并拼接
            pos = torch.cat([pos] * repeat_times + [pos[:remainder]], dim=0)
            # print(pos.shape)

        features = features.clone()
        # print(os.path.join(self.idx, slide_id + '.pt'))
        # print(full_path)
        # print(pos_len, feat_len)


        # features = features[idx]
        # pos = pos[idx]

        # print(full_path)
        # print(os.path.join(self.pos, slide_id + '.pt'))
        # print(pos.shape, features.shape)

        # print(pos.shape, features.shape)
        assert pos.shape[0] == features.shape[0]
        if self.data_set == 'train':
            index = [x for x in range(features.shape[0])]
            random.shuffle(index)
            features = features[index]
            pos = pos[index]

        assert pos.shape[0] == features.shape[0]
        return features, pos, label,slide_id


data_root = '/data/smb/syh/WSI_cls/'
dataset='cam16'
feat='res50'
k=64
ckpt='ckpt/cam16/0/Friday_06_December_2024_15h_53m_10s/epoch=130-val_acc=0.8730.ckpt'

if feat == 'uni':
    feat_ext = 'feat_clam_uni'
elif feat == 'res50':
    feat_ext = 'feat_clam_resnet50'

os.makedirs('grad-cam-out', exist_ok=True)
dataset_root = os.path.join(data_root, dataset, feat_ext)
h5_dir = os.path.join(data_root, dataset, 'feat_clam_resnet50', 'patch_size_256_at_mag_20', 'h5_files')
img_dir = os.path.join(data_root, dataset, 'img')

test_dataset = WSIDataset(dataset_root, 'test', 0, 'cam16')
checkpoint = torch.load(ckpt, map_location=torch.device('cpu'))
state_dict = checkpoint['state_dict']

state_dict={k.replace('model.',''):v for k,v in state_dict.items()}
from models.mixer import MlpMixer
            # model = MlpMixer(input_dim=1024, hidden_dim=512, scale=32768)
model = MlpMixer(input_dim=1024, hidden_dim=512, scale=512)
t=torch.rand([10,1026])
model(t)
model.load_state_dict(state_dict)

model(t)
cam = GradCAM(model=model, target_layers=[model.intra_region3.intra_fc[0]],reshape_transform=lambda x: x.unsqueeze(0))  # 根据设备调整 use_cuda


# 生成 CAM 热力图
targets = None  # 默认使用模型预测的类别

def transform_input_tensor(input_tensor):
    input_tensor = input_tensor.unsqueeze(0)
    return input_tensor

for features, coords, label,slide_id in test_dataset:
    batch=torch.concat([features,coords],dim=1)
    print(features.shape, coords.shape,batch.shape, label,slide_id)
    

    h5_path = os.path.join(h5_dir, slide_id + '.h5')
    img_path = os.path.join(img_dir, slide_id+'.tif')
    
    img_org, downsample_factor = read_tif(img_path) #RGB
    img_org = img_org[:, :, ::-1] #BGR
    
    model(batch)
    
    # img, img_org = draw_points(img_path, coords, k)

    showArrayOrTensor(img_org,'img') 
    showArrayOrTensor(coords,'coords')
    
    grayscale_cam = cam(batch,None)
    print(grayscale_cam[0])
    # 将热力图叠加到原图
    
    
    break


    # cv2.imwrite('tmp/{}.jpg'.format(slide_id), img)
    # cv2.imwrite('tmp/{}_org.jpg'.format(slide_id), img_org)
    break

Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=1, bias=True)
)
use_pos:  True
mod:  6.283185307179586
Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
)
use_pos:  False
Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
)
use_pos:  False


TypeError: MlpMixer.forward() takes 1 positional argument but 2 were given

In [7]:
import cv2
import numpy as np
heatmap = np.random.randint(0, 255, (512, 512), dtype=np.uint8)

colormap=cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
cv2.imwrite('temp/theatmap.jpg',colormap)


True

In [32]:
from models.mixer import MlpMixer
            # model = MlpMixer(input_dim=1024, hidden_dim=512, scale=32768)
model = MlpMixer(input_dim=1024, hidden_dim=512, scale=512)
t=torch.rand([10,1026])
model(t)
    



Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=1, bias=True)
)
use_pos:  True
mod:  6.283185307179586
Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
)
use_pos:  False
Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
)
use_pos:  False


TypeError: MlpMixer.forward() takes 1 positional argument but 2 were given